(course-alzheimer-final-project)=

# Path A - Module 54: Final Project: From Sequence to Analysis

This capstone brings the course tools together around the Amyloid-Beta fibril
and the candidate peptide **KLVFF**. Keep the distinction between constructing a
candidate configuration and demonstrating a biological effect.

## Your Final Mission
Prepare and assemble a candidate complex, then characterize its geometry.
The interface recipe in Step 5 has bounded analytic regression controls.
Steps 1–4 remain a workflow outline: their complete download, repair, capping,
assembly and parameterization route is not execution-qualified here (see [#593](https://github.com/uibcdf/molsysmt/issues/593)). Solvation
does not dock the candidate or establish a realistic binding pose. Do not treat
this notebook as a validated simulation-ready preparation protocol.


## Step 1: Target Preparation
Load the fibril, audit it, and repair it. Make sure it is physically sound.

In [ ]:
import molsysmt as msm

# 1. Load and Audit
fibril = msm.convert('pdb:2BEG', to_form='molsysmt.MolSys')
msm.build.add_missing_heavy_atoms(fibril)
msm.build.add_missing_hydrogens(fibril)
print("Target prepared and repaired.")

## Step 2: Therapeutic Synthesis
Synthesize the KLVFF peptide with appropriate cappings and initial conformation.

In [ ]:
# 2. Build the peptide
peptide = msm.build.build_peptide('KLVFF', n_terminal='ACE', c_terminal='NME', 
                                   configuration='beta-strand')
msm.build.add_missing_hydrogens(peptide)
print("Therapeutic peptide synthesized.")

## Step 3: Systems Assembly & Solvation
Join the prepared dry partners and add solvent. The target and candidate can
both be classified as peptides, so molecular type is not their identity.
Record explicit atom-index populations in merge order instead.

This outline uses ordinary OpenMM solvation, which retains the source atoms as
the leading output prefix. The ranges below therefore survive that operation
and the following coordinate-only minimization. Any extraction, reordering,
atom removal or different preparation route needs its own correspondence map.
Only proceed with dry partners placed in the intended common Cartesian image.


In [ ]:
# 3. Merge and Solvate
n_target_atoms = msm.get(fibril, n_atoms=True)
n_candidate_atoms = msm.get(peptide, n_atoms=True)
target_atom_indices = list(range(n_target_atoms))
candidate_atom_indices = list(range(n_target_atoms, n_target_atoms + n_candidate_atoms))
complex = msm.merge([fibril, peptide])
final_complex = msm.build.solvate(
    complex, engine='OpenMM', box_shape='cubic', clearance='1.0 nm',
    ionic_strength='150 mM'
)
print(f"Complex assembled and solvated with {msm.get(final_complex, n_atoms=True)} atoms.")


## Step 4: Physics Relaxation
Use a parameterized and spatially assembled complex. Retain the independently minimized result for later analysis. Local optimization alone does not ensure that clashes are absent or establish binding stability.

The independent native workflow below differs from a live engine object: unsupported engine-copy requests are rejected before optimization rather than changing the input before failing.


In [ ]:
# 4. Energy Minimization
final_complex = msm.molecular_mechanics.potential_energy_minimization(final_complex)
print("Energy relaxation completed.")

## Step 5: Geometric Interface Characterization

Analyze structure index 0 with the target and candidate populations declared
above. First extract their union, excluding solvent and other background atoms.
Then extract each partner independently: `get_sasa(selection=...)` only filters
outputs and would keep the other atoms as occluders.

At the same coordinates, with the same native van der Waals radii, a **0.14 nm
probe radius** and **960 sphere points per atom**, calculate
$\Delta SASA = SASA(A) + SASA(B) - SASA(A\cup B)$.
Report both the **total SASA loss** and its half, the **mean interface area**.
The half-area convention follows the [PDBePISA interface-area definition](https://www.ebi.ac.uk/pdbe/prot_int/pi_tips.html).
This composition uses MolSysMT's existing Shrake–Rupley calculation
([Shrake and Rupley, 1973](https://doi.org/10.1016/0022-2836(73)90011-9)); it
does not execute PISA or reproduce its radii, assembly analysis or energetics.

The recipe explicitly uses **Cartesian geometry** (`pbc=False`) for SASA and
hydrogen-bond screening. Both partners must be whole and in the intended common
image; it does not reconstruct a periodic interface. Exclude explicit solvent
from this two-partner area definition. Isolated-partner calculations retain
the bound coordinates and perform no separate relaxation. Results depend on
radii, probe, sampled structure and quadrature; study sampling convergence for
quantitative applications.

`get_area_buried` is a Rose reference-table lookup and `get_buried_fraction`
is a Janin reference-table lookup. Neither calculates this geometric interface.
The hydrogen-bond count below uses the explicit-H distance criterion with a
0.23 nm H–acceptor cutoff. Its interpretation requires the supplied hydrogen
inventory. Neither that count nor SASA loss proves binding stability, affinity,
free energy, equilibration or a favorable biological effect. See
{ref}`Tutorial_Get_sasa` and {ref}`Tutorial_Get_buch_hbonds`.


In [ ]:
# 5. Geometric SASA loss and observed H-bond count
if len(target_atom_indices) == 0 or len(candidate_atom_indices) == 0:
    raise ValueError("Both interface populations must be nonempty.")
partner_atom_indices = list(target_atom_indices) + list(candidate_atom_indices)
if len(set(partner_atom_indices)) != len(partner_atom_indices):
    raise ValueError("Interface populations must be unique and disjoint.")

# Extraction preserves this explicit order and gives the dry pair its local indices.
molsys_AB = msm.extract(final_complex, selection=partner_atom_indices, structure_indices=0)
local_A = list(range(len(target_atom_indices)))
local_B = list(range(len(target_atom_indices), len(partner_atom_indices)))
sasa_options = dict(
    engine='MolSysMT', element='atom', probe_radius='0.14 nm',
    n_sphere_points=960, pbc=False, use_gpu=False
)
sasa_A = msm.physchem.get_sasa(msm.extract(molsys_AB, selection=local_A), **sasa_options).sum(axis=1)
sasa_B = msm.physchem.get_sasa(msm.extract(molsys_AB, selection=local_B), **sasa_options).sum(axis=1)
sasa_AB = msm.physchem.get_sasa(molsys_AB, **sasa_options).sum(axis=1)
delta_sasa = sasa_A + sasa_B - sasa_AB
interface_mean_area = delta_sasa / 2

hbonds, _ = msm.interactions.hbonds.get_buch_hbonds(
    molsys_AB, selection=local_A, selection_2=local_B,
    structure_indices=0, distance_threshold='0.23 nm', pbc=False
)
n_hbonds = len(hbonds[0])
puw = msm.pyunitwizard
print("CONFIGURATION REPORT (structure index 0):")
print(f" - Total SASA loss (nm²): {puw.get_value(delta_sasa, to_unit='nm**2')[0]:.6f}")
print(f" - Mean interface area (nm²): {puw.get_value(interface_mean_area, to_unit='nm**2')[0]:.6f}")
print(f" - H-bonds matching the distance criterion: {n_hbonds}")


## Final Words

You have combined preparation, coordinate editing and explicit geometric
observables in the **Path of the Alzheimer's Architect**. The bounded interface
controls validate the recipe's surface definition, units and selections; they
do not qualify the full biological preparation or demonstrate inhibition.
Keep the exact prepared model, partner maps, methods and parameters with any
reported observations, and validate the complete workflow before using it.
